In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split

from sklearn.metrics import precision_score,recall_score,f1_score,average_precision_score, roc_auc_score, confusion_matrix, classification_report,precision_recall_curve

In [ ]:
data_path = Path("../data/creditcard.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print(df.head())

In [4]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (284807, 30)
y shape: (284807,)


In [5]:
# -----------------------------
# Train / Validation / Test Split
# -----------------------------

# First split: 80% development, 20% final held-out test
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Second split: 75% train, 25% validation of development data
# Final proportions:
# Train      = 60%
# Validation = 20%
# Test       = 20%
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    stratify=y_dev,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\nClass distribution:")
print("Train:")
print(y_train.value_counts())

print("\nValidation:")
print(y_val.value_counts())

print("\nTest:")
print(y_test.value_counts())

X_train: (170883, 30)
X_val  : (56962, 30)
X_test : (56962, 30)

Class distribution:
Train:
Class
0    170588
1       295
Name: count, dtype: int64

Validation:
Class
0    56863
1       99
Name: count, dtype: int64

Test:
Class
0    56864
1       98
Name: count, dtype: int64


In [6]:
from xgboost import XGBClassifier

# Calculate class imbalance from training data only
scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

print("scale_pos_weight:", scale_pos_weight)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost trained successfully.")

scale_pos_weight: 578.264406779661
XGBoost trained successfully.


In [7]:
# ---------------------------------
# Step 4: Predict on Validation Set
# ---------------------------------

xgb_val_probability = xgb_model.predict_proba(X_val)[:, 1]

print("Number of validation probabilities:", len(xgb_val_probability))

print("\nFirst 10 validation probabilities:")
print(xgb_val_probability[:10])

Number of validation probabilities: 56962

First 10 validation probabilities:
[9.1583197e-06 3.6752508e-06 4.2600361e-05 4.5911338e-06 5.0007325e-06
 2.0606476e-05 6.4180585e-06 6.6535213e-06 2.7747089e-06 1.4776452e-06]


In [8]:
xgb_probability = xgb_model.predict_proba(X_test)[:, 1]

print("Number of probabilities:", len(xgb_probability))
print("First 10 probabilities:")
print(xgb_probability[:10])

Number of probabilities: 56962
First 10 probabilities:
[1.6516375e-06 4.1474646e-06 2.5723964e-05 5.2935616e-07 1.1684333e-04
 1.5108441e-06 1.2833416e-06 4.1900034e-06 7.4857303e-06 1.1494247e-05]


In [9]:
# ---------------------------------
# Step 5: Validation PR-AUC
# ---------------------------------

val_pr_auc = average_precision_score(
    y_val,
    xgb_val_probability
)

print(f"Validation PR-AUC: {val_pr_auc:.4f}")

Validation PR-AUC: 0.8276


In [10]:
# ---------------------------------
# Step 6A: Cost Function
# ---------------------------------

COST_FALSE_POSITIVE = 100
COST_FALSE_NEGATIVE = 5000

print("False Positive cost: ₹", COST_FALSE_POSITIVE)
print("False Negative cost: ₹", COST_FALSE_NEGATIVE)


def cost_at_threshold(
    y_true,
    y_proba,
    threshold,
    cost_fp,
    cost_fn
):
    y_pred = (y_proba >= threshold).astype(int)

    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tp = np.sum((y_pred == 1) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))

    total_cost = (
        fp * cost_fp +
        fn * cost_fn
    )

    return total_cost, fp, fn, tp, tn

False Positive cost: ₹ 100
False Negative cost: ₹ 5000


In [11]:
# ---------------------------------
# Step 6B: Check Default Threshold
# ---------------------------------

cost, fp, fn, tp, tn = cost_at_threshold(
    y_val,
    xgb_val_probability,
    0.50,
    COST_FALSE_POSITIVE,
    COST_FALSE_NEGATIVE
)

print("Threshold:", 0.50)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)
print("True Negatives:", tn)
print("Total Validation Cost: ₹", cost)

Threshold: 0.5
False Positives: 9
False Negatives: 22
True Positives: 77
True Negatives: 56854
Total Validation Cost: ₹ 110900


In [12]:
# ---------------------------------
# Step 6C: Threshold Sweep
# ---------------------------------

thresholds = np.arange(0.01, 1.00, 0.01)

results = []

for threshold in thresholds:

    cost, fp, fn, tp, tn = cost_at_threshold(
        y_val,
        xgb_val_probability,
        threshold,
        COST_FALSE_POSITIVE,
        COST_FALSE_NEGATIVE
    )

    results.append({
        "threshold": round(float(threshold), 2),
        "total_cost": cost,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
        "true_negatives": tn
    })

cost_df = pd.DataFrame(results)

print("Threshold sweep completed.")
print("\nLowest-cost thresholds:")
print(
    cost_df
    .sort_values("total_cost")
    .head(10)
    .to_string(index=False)
)

Threshold sweep completed.

Lowest-cost thresholds:
 threshold  total_cost  false_positives  false_negatives  true_positives  true_negatives
      0.03       93600               36               18              81           56827
      0.02       95400               54               18              81           56809
      0.01       99400               94               18              81           56769
      0.17      101700               17               20              79           56846
      0.16      101700               17               20              79           56846
      0.15      101700               17               20              79           56846
      0.14      101700               17               20              79           56846
      0.13      101700               17               20              79           56846
      0.12      101800               18               20              79           56845
      0.11      102000               20               20  

In [13]:
# ---------------------------------
# Step 6D: Select and Freeze Threshold
# ---------------------------------

best_row = cost_df.loc[
    cost_df["total_cost"].idxmin()
]

optimal_threshold = float(
    best_row["threshold"]
)

print("=" * 55)
print("VALIDATION COST OPTIMIZATION")
print("=" * 55)

print(f"Optimal threshold : {optimal_threshold:.2f}")
print(f"Validation cost   : ₹{best_row['total_cost']:,.0f}")
print(f"False positives   : {int(best_row['false_positives'])}")
print(f"False negatives   : {int(best_row['false_negatives'])}")
print(f"True positives    : {int(best_row['true_positives'])}")
print(f"True negatives    : {int(best_row['true_negatives'])}")

VALIDATION COST OPTIMIZATION
Optimal threshold : 0.03
Validation cost   : ₹93,600
False positives   : 36
False negatives   : 18
True positives    : 81
True negatives    : 56827


In [14]:
# Freeze the selected threshold.
# This value will NOT be optimized using the test set.

FINAL_THRESHOLD = optimal_threshold

print(f"Frozen final threshold: {FINAL_THRESHOLD:.2f}")

Frozen final threshold: 0.03


In [15]:
# ---------------------------------
# Step 7A: Predict on Final Test Set
# ---------------------------------

xgb_test_probability = xgb_model.predict_proba(X_test)[:, 1]

print("Number of test probabilities:", len(xgb_test_probability))

print("\nFirst 10 test probabilities:")
print(xgb_test_probability[:10])

Number of test probabilities: 56962

First 10 test probabilities:
[1.6516375e-06 4.1474646e-06 2.5723964e-05 5.2935616e-07 1.1684333e-04
 1.5108441e-06 1.2833416e-06 4.1900034e-06 7.4857303e-06 1.1494247e-05]


In [16]:
# ---------------------------------
# Step 7B: Apply Frozen Threshold
# ---------------------------------

test_prediction = (
    xgb_test_probability >= FINAL_THRESHOLD
).astype(int)

print("Frozen threshold:", FINAL_THRESHOLD)

print("\nNumber of transactions flagged as fraud:",test_prediction.sum())
print()

print("\nNumber of transactions not flagged:",(test_prediction == 0).sum())

Frozen threshold: 0.03

Number of transactions flagged as fraud: 135


Number of transactions not flagged: 56827


In [17]:
# ---------------------------------
# Step 7C: Final Test Evaluation
# ---------------------------------

test_precision = precision_score(
    y_test,
    test_prediction,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    test_prediction,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    test_prediction,
    zero_division=0
)

test_pr_auc = average_precision_score(
    y_test,
    xgb_test_probability
)

test_roc_auc = roc_auc_score(
    y_test,
    xgb_test_probability
)

print("=" * 55)
print("FINAL HELD-OUT TEST METRICS")
print("=" * 55)

print(f"Frozen Threshold : {FINAL_THRESHOLD:.2f}")
print(f"Precision        : {test_precision:.4f}")
print(f"Recall           : {test_recall:.4f}")
print(f"F1 Score         : {test_f1:.4f}")
print(f"PR-AUC           : {test_pr_auc:.4f}")
print(f"ROC-AUC          : {test_roc_auc:.4f}")

FINAL HELD-OUT TEST METRICS
Frozen Threshold : 0.03
Precision        : 0.6222
Recall           : 0.8571
F1 Score         : 0.7210
PR-AUC           : 0.8681
ROC-AUC          : 0.9793


In [18]:
# ---------------------------------
# Step 7D: Final Confusion Matrix
#          + FP/FN + Test Cost
# ---------------------------------

test_cm = confusion_matrix(
    y_test,
    test_prediction
)

test_tn, test_fp, test_fn, test_tp = test_cm.ravel()

test_cost = (
    test_fp * COST_FALSE_POSITIVE
    + test_fn * COST_FALSE_NEGATIVE
)

print("=" * 55)
print("FINAL HELD-OUT TEST COST EVALUATION")
print("=" * 55)

print("\nConfusion Matrix:")
print(test_cm)

print("\nConfusion Matrix Breakdown:")
print(f"True Negatives  (TN): {test_tn}")
print(f"False Positives (FP): {test_fp}")
print(f"False Negatives (FN): {test_fn}")
print(f"True Positives  (TP): {test_tp}")

print("\nCost Configuration:")
print(f"False Positive Cost: ₹{COST_FALSE_POSITIVE}")
print(f"False Negative Cost: ₹{COST_FALSE_NEGATIVE}")

print(f"\nFinal Test Cost: ₹{test_cost:,.0f}")

FINAL HELD-OUT TEST COST EVALUATION

Confusion Matrix:
[[56813    51]
 [   14    84]]

Confusion Matrix Breakdown:
True Negatives  (TN): 56813
False Positives (FP): 51
False Negatives (FN): 14
True Positives  (TP): 84

Cost Configuration:
False Positive Cost: ₹100
False Negative Cost: ₹5000

Final Test Cost: ₹75,100


In [19]:
# ---------------------------------
# Step 7E: Compare Default vs
#          Frozen Threshold
#          on Final Test Set
# ---------------------------------

# Default threshold
default_threshold = 0.50

default_cost, default_fp, default_fn, default_tp, default_tn = (
    cost_at_threshold(
        y_test,
        xgb_test_probability,
        default_threshold,
        COST_FALSE_POSITIVE,
        COST_FALSE_NEGATIVE
    )
)

# Frozen validation-selected threshold
frozen_cost, frozen_fp, frozen_fn, frozen_tp, frozen_tn = (
    cost_at_threshold(
        y_test,
        xgb_test_probability,
        FINAL_THRESHOLD,
        COST_FALSE_POSITIVE,
        COST_FALSE_NEGATIVE
    )
)

cost_reduction = default_cost - frozen_cost
cost_reduction_percent = (
    cost_reduction / default_cost * 100
)

print("=" * 60)
print("DEFAULT vs FROZEN THRESHOLD — FINAL TEST SET")
print("=" * 60)

print("\nDEFAULT THRESHOLD")
print(f"Threshold       : {default_threshold:.2f}")
print(f"False Positives : {default_fp}")
print(f"False Negatives : {default_fn}")
print(f"True Positives  : {default_tp}")
print(f"True Negatives  : {default_tn}")
print(f"Total Cost      : ₹{default_cost:,.0f}")

print("\nFROZEN THRESHOLD")
print(f"Threshold       : {FINAL_THRESHOLD:.2f}")
print(f"False Positives : {frozen_fp}")
print(f"False Negatives : {frozen_fn}")
print(f"True Positives  : {frozen_tp}")
print(f"True Negatives  : {frozen_tn}")
print(f"Total Cost      : ₹{frozen_cost:,.0f}")

print("\nCOST IMPROVEMENT")
print(f"Cost Reduction  : ₹{cost_reduction:,.0f}")
print(f"Reduction (%)   : {cost_reduction_percent:.2f}%")

DEFAULT vs FROZEN THRESHOLD — FINAL TEST SET

DEFAULT THRESHOLD
Threshold       : 0.50
False Positives : 10
False Negatives : 16
True Positives  : 82
True Negatives  : 56854
Total Cost      : ₹81,000

FROZEN THRESHOLD
Threshold       : 0.03
False Positives : 51
False Negatives : 14
True Positives  : 84
True Negatives  : 56813
Total Cost      : ₹75,100

COST IMPROVEMENT
Cost Reduction  : ₹5,900
Reduction (%)   : 7.28%


In [20]:
# ---------------------------------
# Step 8: Save Final Model + Config
# ---------------------------------

from pathlib import Path
import joblib

model_dir = Path("../models")
model_dir.mkdir(exist_ok=True)

# Save the exact XGBoost model used
# for the final held-out test evaluation.
joblib.dump(
    xgb_model,
    model_dir / "final_xgb_model.pkl"
)

# Save the frozen risk policy.
final_risk_config = {
    "model": "XGBoost",
    "threshold": float(FINAL_THRESHOLD),
    "false_positive_cost": float(COST_FALSE_POSITIVE),
    "false_negative_cost": float(COST_FALSE_NEGATIVE)
}

joblib.dump(
    final_risk_config,
    model_dir / "final_risk_config.pkl"
)

print("=" * 55)
print("FINAL MODEL ARTIFACTS SAVED")
print("=" * 55)

print(
    "Final model:",
    (model_dir / "final_xgb_model.pkl").resolve()
)

print(
    "Final config:",
    (model_dir / "final_risk_config.pkl").resolve()
)

print("\nFinal configuration:")
print(final_risk_config)

FINAL MODEL ARTIFACTS SAVED
Final model: C:\Users\HP\OneDrive\Desktop\Fraud Risk Detect\models\final_xgb_model.pkl
Final config: C:\Users\HP\OneDrive\Desktop\Fraud Risk Detect\models\final_risk_config.pkl

Final configuration:
{'model': 'XGBoost', 'threshold': 0.03, 'false_positive_cost': 100.0, 'false_negative_cost': 5000.0}
